# Lab 2.2 · Repair, cap and a human queue

Day 2 · Session 2 (Validation, retry and fallback) · 25 min.

One message in five failed the schema on the first try in Lab 2.1. You turn each validation error into a repair
message, cap the repairs, and send what still fails to a human queue, so that no rider message is lost and nothing
invalid reaches the ticket service. Then you measure against Darb's 95% target and write two tests.

**Reference state.** `wasil.RiderMessage` (the schema of Lab 2.1) and the same 50 messages (20 in quick mode).

**Tasks**
- [ ] 1 · A validation error as a repair message (6')
- [ ] 2 · The repair loop with a cap (7')
- [ ] 3 · A human queue for what still fails (6')
- [ ] 4 · Measure against the target and test (6')
- [ ] ✱ Stretch · All 300 messages, by language

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ae-203.git"
    if not Path("/content/llm-applications").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/llm-applications"], check=True)
    os.chdir("/content/llm-applications/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    try:
        from google.colab import userdata  # the key from Colab's Secrets panel
        os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY"))
    except Exception:
        pass
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import json
import shutil
import pandas as pd
from pydantic import ValidationError

import llm
import darb as D
from wasil import RiderMessage, EXTRACT_SYSTEM

OUT = Path("artifacts/lab22")
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
SAMPLE = D.messages()[: llm.pick(50, 20)]
SCHEMA = RiderMessage.model_json_schema()
FMT = {"type": "json_schema", "json_schema": {"name": "RiderMessage", "schema": SCHEMA}}
print(llm.describe(), "|", len(SAMPLE), "messages")


def first_messages(m: dict) -> list[dict]:
    return [{"role": "system", "content": EXTRACT_SYSTEM.format(received=m["received"], schema=json.dumps(SCHEMA))},
            {"role": "user", "content": m["text"]}]


def validate(reply: str) -> RiderMessage:
    s, e = reply.find("{"), reply.rfind("}")
    return RiderMessage.model_validate_json(reply[s:e + 1] if s >= 0 else reply)

## 1 · A validation error as a repair message (6')

Pydantic's error lists each failing field, what was wrong and the value it got. That is all a model needs to fix it.

✏️ **Exercise** · Write `feedback(err)` that returns one short message: "That JSON is invalid: " followed by one line
per error (`field: message (got value)`), then "Reply with the corrected JSON only."

In [ ]:
def feedback(err: ValidationError) -> str:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    validate('{"route": "B٢٧", "date": "the twelfth of November", "complaint_type": "late", "refund_requested": false}')
except ValidationError as err:
    try:
        msg = feedback(err)
    except NotImplementedError:
        msg = "That JSON is invalid:\n" + str(err) + "\nReply with the corrected JSON only."
print(msg)
llm.check("route" in msg and "date" in msg and "complaint_type" in msg, "the message names all three fields")

## 2 · The repair loop with a cap (7')

✏️ **Exercise** · Write `extract(m, max_repairs=2)`. Ask once; on a validation error, append the reply as an assistant
message and `feedback(err)` as a user message, and ask again, at most `max_repairs` times. Return `(object, attempts)`,
or raise the last `ValidationError` when the cap is reached. A reply with no JSON at all counts as invalid too.

In [ ]:
def extract(m: dict, max_repairs: int = 2) -> tuple[RiderMessage, int]:
    msgs = first_messages(m)
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
hard = next(m for m in SAMPLE if any("٠" <= ch <= "٩" for ch in m["text"]))
try:
    obj, attempts = extract(hard)
except NotImplementedError:
    obj, attempts = RiderMessage(**hard["gold"]), 1
print(hard["text"], "\n->", obj.model_dump(mode="json"), "after", attempts, "repair(s)")
llm.check(obj.route == hard["gold"]["route"], "repaired to the right route")

## 3 · A human queue for what still fails (6')

A message that still fails after the cap goes to a person, with the error, never to the ticket service.

✏️ **Exercise** · Write `process(sample)`: for each message call `extract`; on success record the object and the
attempts; on a `ValidationError` append `{"id", "text", "error"}` to `OUT / "human_queue.jsonl"`. Return a DataFrame
with one row per message: id, lang, outcome (`first_try`, `repaired` or `queued`), attempts, fields right.

In [ ]:
def process(sample: list[dict]) -> pd.DataFrame:
    rows = []
    queue = OUT / "human_queue.jsonl"
    for m in sample:
        # ✏️ YOUR CODE HERE
        raise NotImplementedError("✏️ Exercise: write your solution here")
    return pd.DataFrame(rows)

In [ ]:
try:
    res = process(SAMPLE)
except NotImplementedError:
    res = pd.DataFrame([{"id": m["id"], "lang": m["lang"], "outcome": "first_try", "attempts": 0, "fields_right": 4}
                        for m in SAMPLE])
print(res.outcome.value_counts().to_string())
queued = (OUT / "human_queue.jsonl").read_text(encoding="utf-8").splitlines() if (OUT / "human_queue.jsonl").exists() else []
for line in queued:
    print("queued:", json.loads(line)["text"][:90])

## 4 · Measure against the target and test (6')

The valid-output rate counts what reaches the ticket service valid, after repairs. The queue is the fallback path.

✏️ **Exercise** · Compute `valid_rate` (first try plus repaired, over all) and write `test_repairs_a_date_in_words`
and `test_queues_what_cannot_be_fixed` against the two supplied messages.

In [ ]:
def test_repairs_a_date_in_words():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


def test_queues_what_cannot_be_fixed():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
valid_rate = (res.outcome != "queued").mean()
first_rate = (res.outcome == "first_try").mean()
print(f"first try {first_rate:.0%} -> after repairs {valid_rate:.0%} (target 95%) | queued {int((res.outcome == 'queued').sum())}"
      f" | mean attempts {res.attempts.mean():.2f}")
print(llm.LEDGER)
for t in (test_repairs_a_date_in_words, test_queues_what_cannot_be_fixed):
    try:
        t()
        print("pass", t.__name__)
    except NotImplementedError:
        print("not written", t.__name__)
llm.check(valid_rate >= first_rate, "repairs never lower the valid rate")

🔍 **Look at** · the queued messages. If they carry instructions to the assistant, the queue is doing a second job:
Day 4 flags them before they reach a model at all.

## ✱ Stretch · All 300 messages, by language

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    if not llm.QUICK:
        stretch()
except NotImplementedError:
    print("✱ stretch not attempted")

## Checkpoint

1. Why send the model its own invalid reply as well as the error?
<details><summary>Answer</summary>So it corrects that reply instead of starting again, and the error points at
what to change in it.</details>

2. Why cap the repairs at two?
<details><summary>Answer</summary>Each repair is another paid call and more latency; a reply still invalid after two
rarely gets better, and a person handles it faster.</details>

3. The valid rate after repairs is 98% and field accuracy 90%. Is the extraction good enough?
<details><summary>Answer</summary>Not by validity alone: valid but wrong fields reach the ticket service. Both numbers
go into the report, against stated targets.</details>

**Carry forward.** `wasil.extract` is this loop; the assistant of Day 4 uses it and keeps the queue.